# 1. Limpeza e transformação dos dados

**EBTT-UFPB — pipeline de dados**

Este notebook carrega os Parquets brutos em `dados/`, **limpa e integra** as tabelas, define a **variável-alvo** (`target`), calcula **features temporais** (histórico por área de conhecimento, CH integralizada acumulada, etc.) e **exporta** um CSV único para EDA e modelagem.

---

### Roteiro

1. **Configuração** — bibliotecas e constantes.
2. **Carregamento e inspeção** — dimensões, tipos, nulos e checagens exploratórias.
3. **Limpeza por fonte** — discentes, docentes (agregado departamento × ano), componentes, cursos.
4. **Integração** — junção no nível da matrícula (sem repetir docentes por linha nesta versão).
5. **Variável-alvo** — mapeamento de situações em sucesso/insucesso e filtros.
6. **Engenharia de features** — ordenação temporal, histórico por área, idade, primeiro período.
7. **Exportação** — `dados/df_modelo_tratado.csv`.

**Próximo passo:** `02_analise_exploratoria.ipynb`


## Definição do problema

Este projeto busca um modelo capaz de **predizer o desempenho acadêmico dos discentes no EBTT-UFPB**, considerando o ecossistema **aluno — professor — currículo**.

Trata-se de **classificação**: identificar padrões que separem, por exemplo, **aprovação** e **reprovação** (e situações equivalentes) em componentes curriculares.

Foram integradas bases institucionais:

- **Discentes** — perfil e histórico agregado ao nível aluno.
- **Matrículas** — trajetória (uma linha por tentativa em disciplina).
- **Componentes curriculares** — estrutura da disciplina.
- **Cursos** — vínculo currículo × disciplina × carga horária do curso.
- **Docentes** — características agregadas por **departamento e ano** (professores ativos naquele ano).

A hipótese é que o desempenho depende da interação entre fatores individuais, docentes e estrutura curricular. O pipeline abaixo prepara uma tabela analítica **no nível da matrícula**, com histórico acumulado **somente até aquela linha** (sem vazamento de futuro).


## 1. Configuração do ambiente

Importamos **pandas** e **numpy** para manipulação tabular e numérica; **json** para o mapa de áreas das disciplinas. `warnings.filterwarnings('ignore')` reduz ruído na execução (ajuste se quiser ver alertas). `SEED` padroniza qualquer aleatoriedade em notebooks posteriores.

In [182]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np

print('Ambiente configurado (limpeza).')
print(f'pandas {pd.__version__}')


Ambiente configurado (limpeza).
pandas 3.0.1


## 2. Carregamento e inspeção dos dados

Os arquivos **Parquet** em `dados/` são lidos em DataFrames. O dicionário `datasets` centraliza nomes para relatar **linhas × colunas** de forma uniforme.

In [183]:
DATA_PATH = 'dados/'

discentes   = pd.read_parquet(DATA_PATH + 'discentes.parquet')
docentes    = pd.read_parquet(DATA_PATH + 'docentes.parquet')
matriculas  = pd.read_parquet(DATA_PATH + 'matriculas.parquet')
componentes = pd.read_parquet(DATA_PATH + 'componentes.parquet')
cursos      = pd.read_parquet(DATA_PATH + 'cursos.parquet')

datasets = {
    'discentes': discentes, 'docentes': docentes,
    'matriculas': matriculas, 'componentes': componentes, 
    'cursos': cursos
}

print(f'{"Dataset":<15} {"Linhas":>10} {"Colunas":>10}')
print('-' * 38)
for nome, df in datasets.items():
    print(f'{nome:<15} {df.shape[0]:>10,} {df.shape[1]:>10}')


Dataset             Linhas    Colunas
--------------------------------------
discentes            7,707         21
docentes               348         11
matriculas         426,753          5
componentes            998         22
cursos               1,310         19


Para cada base, listamos **dtypes**, **percentual de nulos** por coluna (quando houver) e uma **amostra** das três primeiras linhas. Isso orienta decisões de limpeza e chaves de junção.

In [184]:
for nome, df in datasets.items():
    print(f'\n{"="*55}\n  {nome.upper()}\n{"="*55}')
    print(df.dtypes.to_string())
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print('\n  Valores nulos:')
    if len(nulos):
        for col, n in nulos.items():
            print(f'    {col}: {n} ({n/len(df)*100:.1f}%)')
    else:
        print('    Nenhum.')
    df.info()
    display(df.head(3))



  DISCENTES
id_discente                     object
sexo                            object
estado_civil                    object
raca_declarada                  object
discente_nivel                  object
id_curso                         int64
id_curriculo                   float64
id_estrutura_curricular          int64
ano_ingresso                     int64
periodo_ingresso                 int64
status_discente                 object
forma_ingresso                  object
quantidade_membros_familia     float64
ch_integralizada               float64
ch_pendente                    float64
media_geral                    float64
ano_nascimento                 float64
faixa_renda_familiar          category
uf_titulo_eleitor_pb           float64
uf_naturalidade_pb             float64
pais_origem_br                 float64

  Valores nulos:
    id_curriculo: 7707 (100.0%)
    quantidade_membros_familia: 5587 (72.5%)
    ch_integralizada: 1871 (24.3%)
    ch_pendente: 1871 (24.3%)
    medi

,id_discente,sexo,estado_civil,raca_declarada,discente_nivel,id_curso,id_curriculo,id_estrutura_curricular,ano_ingresso,periodo_ingresso,...,forma_ingresso,quantidade_membros_familia,ch_integralizada,ch_pendente,media_geral,ano_nascimento,faixa_renda_familiar,uf_titulo_eleitor_pb,uf_naturalidade_pb,pais_origem_br
0,1169287803480640,M,Solteiro(a),Nao_informado,T,2558681,NaN,25587020,2016,2,...,PROCESSO SELETIVO,NaN,68.0,1432.0,0.93,1994.0,NaN,NaN,1.0,1.0
1,8959954670178323,F,Solteiro(a),Nao_informado,T,1958830,NaN,23647550,2016,2,...,PROCESSO SELETIVO,NaN,NaN,NaN,0.00,1997.0,NaN,1.0,1.0,1.0
2,9718041069456731,F,Solteiro(a),Negra,T,31496451,NaN,319171030,2023,1,...,PROCESSO SELETIVO,NaN,240.0,0.0,9.05,1970.0,NaN,NaN,1.0,1.0



  DOCENTES
id_pessoa              object
sexo                   object
tipo_vinculo           object
sigla_departamento     object
nome_departamento      object
sigla_centro           object
nome_centro            object
situacao               object
cargo                  object
ano_admissao          float64
ano_desligamento      float64

  Valores nulos:
    ano_admissao: 6 (1.7%)
    ano_desligamento: 131 (37.6%)
<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pessoa           348 non-null    object 
 1   sexo                348 non-null    object 
 2   tipo_vinculo        348 non-null    object 
 3   sigla_departamento  348 non-null    object 
 4   nome_departamento   348 non-null    object 
 5   sigla_centro        348 non-null    object 
 6   nome_centro         348 non-null    object 
 7   situacao            348 non-null    obj

,id_pessoa,sexo,tipo_vinculo,sigla_departamento,nome_departamento,sigla_centro,nome_centro,situacao,cargo,ano_admissao,ano_desligamento
0,5986743761005378,F,DE,CPT-DRPAS,"CPT-ETS - DEPARTAMENTO DE REGISTRO, PROMOÇÃO E...",CPT-ETS,CENTRO PROFISSIONAL E TECNOLÓGICO - ESCOLA TÉC...,Colaborador PCCTAE,PROFESSOR ENS BASICO TECN TECNOLOGICO,2020.0,2024.0
1,6486331404889296,M,DE,PROEX,PRÓ-REITORIA DE EXTENSÃO (PROEX),UFPB.,UNIVERSIDADE FEDERAL DA PARAÍBA,Colaborador PCCTAE,PROFESSOR ENS BASICO TECN TECNOLOGICO,2022.0,2025.0
2,9067115124174644,F,DE,CCHSA-DCBS,CCHSA - DEPARTAMENTO DE CIÊNCIAS BÁSICAS E SOC...,CCHSA,CENTRO DE CIÊNCIAS HUMANAS SOCIAIS E AGRÁRIAS ...,Colaborador PCCTAE,PROF DO ENSINO BASICO TEC TECNOLOGICO,2018.0,2019.0



  MATRICULAS
id_discente      object
id_disciplina     int64
ano              object
periodo          object
situacao         object

  Valores nulos:
    Nenhum.
<class 'pandas.DataFrame'>
RangeIndex: 426753 entries, 0 to 426752
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id_discente    426753 non-null  object
 1   id_disciplina  426753 non-null  int64 
 2   ano            426753 non-null  object
 3   periodo        426753 non-null  object
 4   situacao       426753 non-null  object
dtypes: int64(1), object(4)
memory usage: 16.3+ MB


,id_discente,id_disciplina,ano,periodo,situacao
0,9519063299906826,25772,2020,1,APROVADO
1,9519063299906826,25774,2020,1,REPROVADO
2,9519063299906826,25773,2020,1,REPROVADO



  COMPONENTES
id_disciplina                            int64
id_detalhe                               int64
nome                                    object
ch_aula                                  int64
ch_laboratorio                           int64
ch_total                                 int64
cr_aula                                  int64
cr_laboratorio                           int64
cr_estagio                               int64
ch_ead                                   int64
sigla_departamento                      object
nivel_componente_curricular             object
sigla_academica                         object
nome_departamento                       object
sigla_centro                            object
nome_centro                             object
qtd_max_matriculas                       int64
codigo_componente_curricular            object
nome_componete_curricular               object
descricao_tipo_componente_curricular    object
excluir_avaliacao_institucional           boo

,id_disciplina,id_detalhe,nome,ch_aula,ch_laboratorio,ch_total,cr_aula,cr_laboratorio,cr_estagio,ch_ead,...,sigla_academica,nome_departamento,sigla_centro,nome_centro,qtd_max_matriculas,codigo_componente_curricular,nome_componete_curricular,descricao_tipo_componente_curricular,excluir_avaliacao_institucional,ativo
0,18066,2364646,NOÇÕES DE PRIMEIROS SOCORROS,30,0,30,2,0,0,0,...,ETS,CPT-ETS - COORDENAÇÃO ACADÊMICA GERAL DE CURSO...,CPT-ETS,CENTRO PROFISSIONAL E TECNOLÓGICO - ESCOLA TÉC...,1,TETS0001,NOÇÕES DE PRIMEIROS SOCORROS,DISCIPLINA,False,True
1,18069,2364635,NOÇÕES DE ANATOMIA E FISIOLOGIA HUMANA,40,0,40,2,0,0,0,...,ETS,CPT-ETS - COORDENAÇÃO ACADÊMICA GERAL DE CURSO...,CPT-ETS,CENTRO PROFISSIONAL E TECNOLÓGICO - ESCOLA TÉC...,1,TETS0002,NOÇÕES DE ANATOMIA E FISIOLOGIA HUMANA,DISCIPLINA,False,True
2,18070,2741620,"NOÇÕES DE MICROBIOLOGIA, PARASITOLOGIA E IMUNO...",10,30,40,2,0,0,0,...,ETS,CPT-ETS - COORDENAÇÃO ACADÊMICA GERAL DE CURSO...,CPT-ETS,CENTRO PROFISSIONAL E TECNOLÓGICO - ESCOLA TÉC...,1,TETS0003,"NOÇÕES DE MICROBIOLOGIA, PARASITOLOGIA E IMUNO...",DISCIPLINA,False,True



  CURSOS
id_curso                            int64
curso_nome                         object
curso_unidade_nome                 object
campus                             object
curso_ativo                          bool
curso_grande_area_conhecimento    float64
id_estrutura_curricular             int64
codigo_emec                       float64
turno_estrutura_curricular         object
periodo                            object
estrutura_curricular_ativo           bool
carga_horaria                       int64
id_modulo                           int64
codigo                             object
modulo                             object
carga_horaria_modulo                int64
ordem_oferta_modulo               float64
id_disciplina                       int64
codigo_disciplina                  object

  Valores nulos:
    curso_grande_area_conhecimento: 1310 (100.0%)
    codigo_emec: 1310 (100.0%)
    ordem_oferta_modulo: 617 (47.1%)
<class 'pandas.DataFrame'>
RangeIndex: 1310 entries, 0 t

,id_curso,curso_nome,curso_unidade_nome,campus,curso_ativo,curso_grande_area_conhecimento,id_estrutura_curricular,codigo_emec,turno_estrutura_curricular,periodo,estrutura_curricular_ativo,carga_horaria,id_modulo,codigo,modulo,carga_horaria_modulo,ordem_oferta_modulo,id_disciplina,codigo_disciplina
0,31102902,TÉCNICO DE NÍVEL MÉDIO EM GUIA DE TURISMO INTE...,CCHSA - CAVN - COLÉGIO AGRÍCOLA VIDAL DE NEGRE...,Bananeiras,True,NaN,311029220,NaN,Noturno,N,True,2000,31102776,MOD0069,BASE NACIONAL COMUM CURRICULAR (BNCC),1200,1.0,27342,CAVN00112
1,31102902,TÉCNICO DE NÍVEL MÉDIO EM GUIA DE TURISMO INTE...,CCHSA - CAVN - COLÉGIO AGRÍCOLA VIDAL DE NEGRE...,Bananeiras,True,NaN,311029220,NaN,Noturno,N,True,2000,31102776,MOD0069,BASE NACIONAL COMUM CURRICULAR (BNCC),1200,1.0,28736,CAVN00165
2,31102902,TÉCNICO DE NÍVEL MÉDIO EM GUIA DE TURISMO INTE...,CCHSA - CAVN - COLÉGIO AGRÍCOLA VIDAL DE NEGRE...,Bananeiras,True,NaN,311029220,NaN,Noturno,N,True,2000,31102776,MOD0069,BASE NACIONAL COMUM CURRICULAR (BNCC),1200,1.0,28737,CAVN00166


**Matrículas — situação e anos.** A coluna `situacao` concentra o desfecho de cada vínculo; `ano` define a referência temporal para joins com agregados de docentes.

In [185]:
print('Situacoes nas matriculas:')
print(matriculas['situacao'].value_counts().to_string())
print(f'\nAnos: {sorted(matriculas["ano"].unique())}')


Situacoes nas matriculas:
situacao
APROVADO               318849
CANCELADO               32931
REPROVADO               25175
EXCLUIDA                14846
REP. MEDIA E FALTA      12986
MATRICULADO             11770
TRANCADO                 4560
APROVEITADO              4197
DISPENSADO                470
REP. FALTA                411
DESISTENCIA               322
TRANSFERIDO               221
INDEFERIDO                  8
APROV. C/ DISTINÇÃO         7

Anos: ['2004', '2005', '2006', '2009', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [186]:
print(matriculas['ano'].value_counts().to_string())

ano
2023    156103
2024     84018
2020     49157
2018     30385
2017     26443
2022     21629
2021     21459
2019     13881
2016     13405
2015      9942
2014       183
2013        71
2009        24
2025        18
2005        11
2004        10
2006         7
2012         6
2011         1


**Componentes — nomes de disciplina.** Usados depois com `mapeamento_disciplinas.json` para obter `area_conhecimento`. (Coluna original preserva o nome institucional com a grafia do sistema.)

In [187]:
componentes["nome_componete_curricular"].unique()

array(['NOÇÕES DE PRIMEIROS SOCORROS',
       'NOÇÕES DE ANATOMIA E FISIOLOGIA HUMANA',
       'NOÇÕES DE MICROBIOLOGIA, PARASITOLOGIA E IMUNOLOGIA',
       'EDUCAÇÃO EM SAÚDE', 'PSICOLOGIA APLICADA À SAÚDE',
       'ÉTICA EM SAÚDE', 'INTRODUÇÃO À MICROINFORMÁTICA',
       'ANATOMIA E FISIOLOGIA DA CABEÇA E PESCOÇO',
       'ÉTICA PROFISSIONAL E LEGISLAÇÃO TRABALHISTA', 'BIOSSEGURANÇA',
       'ANATOMIA E ESCULTURA DENTAL I E II',
       'ANATOMIA E ESCULTURA DENTAL II', 'MATERIAIS DENTÁRIOS',
       'PRÓTESE FIXA I', 'PRÓTESE FIXA II', 'PRÓTESE FIXA III',
       'PRÓTESE FIXA IV', 'PRÓTESE PARCIAL REMOVÍVEL',
       'PRÓTESE TOTAL I E II', 'INTRODUÇÃO À PRÓTESE DENTÁRIA',
       'PRÓTESE ORTODÔNTICA', 'MOLDES E MODELOS', 'OCLUSÃO DENTAL',
       'ADMINISTRAÇÃO', 'ESTÁGIO SUPERVISIONADO OCLUSÃO',
       'NOÇÕES DE ANATOMIA E FISIOLOGIA DO SISTEMA ESTOMATOGNÁTICO',
       'ÉTICA E LEGISLAÇÃO',
       'BIOSSEGURANÇA E ERGONOMIA NO LABORATÓRIO DE PRÓTESE DENTÁRIA',
       'INICIAÇÃO À PES

**Docentes — visão geral, cargo e situação.** A limpeza principal ocorre na agregação por departamento/ano; estas células apenas exploram a base.

In [188]:
docentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pessoa           348 non-null    object 
 1   sexo                348 non-null    object 
 2   tipo_vinculo        348 non-null    object 
 3   sigla_departamento  348 non-null    object 
 4   nome_departamento   348 non-null    object 
 5   sigla_centro        348 non-null    object 
 6   nome_centro         348 non-null    object 
 7   situacao            348 non-null    object 
 8   cargo               348 non-null    object 
 9   ano_admissao        342 non-null    float64
 10  ano_desligamento    217 non-null    float64
dtypes: float64(2), object(9)
memory usage: 30.0+ KB


In [189]:
docentes['cargo'].value_counts()

cargo
PROFESSOR ENS BASICO TECN TECNOLOGICO       309
PROF DO ENSINO BASICO TEC TECNOLOGICO        36
PROF ENS BAS TEC TECNOLOGICO - VISITANTE      2
PROF ENS BAS TEC TECNOLOGICO-SUBSTITUTO       1
Name: count, dtype: int64

In [190]:
docentes['situacao'].value_counts()

situacao
Ativo Permanente                     142
Aposentado                           121
Instituidor de Pensão                 75
Colaborador PCCTAE                     3
Cedido                                 2
Professor Visitante                    2
Exercicio provisorio                   1
Professor Substituto                   1
Requisitado por Força de Trabalho      1
Name: count, dtype: int64

## 3. Limpeza por fonte e integração

Relação lógica entre tabelas (chaves usadas nos merges):

```
matriculas
  ├── id_discente   ──► discentes   (perfil do aluno)
  └── id_disciplina ──► componentes (currículo)
                            └── sigla_departamento ──► docentes (agregado por dept. × ano)
  └── id_curso, id_estrutura_curricular, id_disciplina ──► cursos
```

A seguir, cada subseção **normaliza uma fonte** e reduz colunas ao necessário para o modelo. Depois, **unimos tudo no nível da matrícula**.

### 3.1. Tabela de discentes

1. **Forma de ingresso** — binária: `PROCESSO SELETIVO` vs `outros` (padronização de texto).
2. **CH integralizada / pendente** — faltas viram zero (interpretação: nada registrado).
3. **Outliers em `quantidade_membros_familia`** — máscara IQR (1,5×); valores extremos são anulados e a coluna recebe **mediana** (robusto).
4. **`faixa_renda_familiar`** — categoria explícita `nulo` onde faltar.
5. **`faixa_membros_familia`** — discretização da composição familiar.
6. **`discente_limpo`** — subconjunto de colunas + `drop_duplicates` por `id_discente`.

In [191]:
m = (
    discentes["forma_ingresso"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("PROCESSO SELETIVO")
)
discentes["forma_ingresso"] = np.where(m, "PROCESSO SELETIVO", "outros")


Verificação da estrutura e distribuição numérica após a primeira transformação.

In [192]:
discentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 7707 entries, 0 to 7706
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   id_discente                 7707 non-null   object  
 1   sexo                        7707 non-null   object  
 2   estado_civil                7707 non-null   object  
 3   raca_declarada              7707 non-null   object  
 4   discente_nivel              7707 non-null   object  
 5   id_curso                    7707 non-null   int64   
 6   id_curriculo                0 non-null      float64 
 7   id_estrutura_curricular     7707 non-null   int64   
 8   ano_ingresso                7707 non-null   int64   
 9   periodo_ingresso            7707 non-null   int64   
 10  status_discente             7707 non-null   object  
 11  forma_ingresso              7707 non-null   str     
 12  quantidade_membros_familia  2120 non-null   float64 
 13  ch_integralizada            5

In [193]:
cols_num = discentes.select_dtypes(include=["number"]).columns
discentes[cols_num].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,id_curso,id_curriculo,id_estrutura_curricular,ano_ingresso,periodo_ingresso,quantidade_membros_familia,ch_integralizada,ch_pendente,media_geral,ano_nascimento,uf_titulo_eleitor_pb,uf_naturalidade_pb,pais_origem_br
count,7.707000e+03,0.0,7.707000e+03,7707.000000,7707.000000,2120.000000,5836.000000,5836.000000,7042.000000,7704.000000,5944.000000,7681.000000,7707.000000
mean,9.753725e+06,NaN,1.396363e+08,2020.281951,1.501881,4.298113,923.295579,513.168609,5.809480,1991.290109,0.754879,0.779586,0.985079
std,1.084091e+07,NaN,1.170914e+08,2.798593,0.500029,25.634208,942.779275,787.616136,3.521833,60.897226,0.430195,0.414553,0.121247
min,1.958830e+06,NaN,1.958833e+07,2015.000000,1.000000,1.000000,15.000000,-510.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1%,1.958830e+06,NaN,1.958833e+07,2015.000000,1.000000,1.000000,15.000000,-60.000000,0.000000,1962.000000,0.000000,0.000000,0.000000
5%,1.958830e+06,NaN,1.997135e+07,2015.000000,1.000000,1.000000,30.000000,-16.000000,0.000000,1972.000000,0.000000,0.000000,1.000000
50%,2.558681e+06,NaN,1.543249e+08,2021.000000,2.000000,3.000000,658.000000,90.000000,7.660000,1996.000000,1.000000,1.000000,1.000000
95%,3.116979e+07,NaN,3.153648e+08,2024.000000,2.000000,6.000000,3360.000000,1985.000000,9.510000,2005.000000,1.000000,1.000000,1.000000
99%,3.812166e+07,NaN,3.839984e+08,2024.000000,2.000000,8.000000,4045.000000,3945.000000,9.920000,2009.000000,1.000000,1.000000,1.000000
max,3.909537e+07,NaN,3.909541e+08,2024.000000,2.000000,890.000000,4105.000000,4030.000000,10.000000,2010.000000,1.000000,1.000000,1.000000


In [194]:
discentes["ch_integralizada"] = discentes["ch_integralizada"].fillna(0)
discentes["ch_pendente"] = discentes["ch_pendente"].fillna(0)

Função auxiliar **IQR** reutilizada também na variável `idade` mais adiante. Aqui limita valores irreais em tamanho da família.

In [195]:
def mascara_outliers_iqr(s, k=1.5, apenas_superior=False):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return pd.Series(False, index=s.index)
    low, high = q1 - k * iqr, q3 + k * iqr
    if apenas_superior:
        return s > high
    return (s < low) | (s > high)

k = 1.5
colunas = ["quantidade_membros_familia"]

for col in colunas:
    if col not in discentes.columns:
        continue
    mask = mascara_outliers_iqr(discentes[col], k=k, apenas_superior=False)
    mask = mask.reindex(discentes.index).fillna(False)
    discentes.loc[mask, col] = np.nan

for col in colunas:
    if col in discentes.columns:
        discentes[col] = discentes[col].fillna(discentes[col].median())


In [196]:
discentes.boxplot(column=["quantidade_membros_familia"], figsize=(10, 4))

<Axes: >

In [197]:
c = discentes['faixa_renda_familiar']
if pd.api.types.is_categorical_dtype(c):
    if 'nulo' not in c.cat.categories:
        c = c.cat.add_categories(['nulo'])
    discentes['faixa_renda_familiar'] = c.fillna('nulo')
else:
    discentes['faixa_renda_familiar'] = c.fillna('nulo')

s = discentes["quantidade_membros_familia"]
faixa = pd.cut(
    s,
    bins=[0, 1, 3, 5, np.inf],
    labels=["1", "2-3", "4-5", "6+"],
    include_lowest=True,
)

discentes["faixa_membros_familia"] = faixa.astype("object").where(s.notna(), "nulo")


In [198]:
colunas_discente = ["id_discente","status_discente" ,"sexo", "ano_nascimento", "estado_civil", "raca_declarada",
                    "id_curso", "id_estrutura_curricular",  'media_geral',
                    "ch_integralizada", "ch_pendente", "faixa_renda_familiar",
                    'periodo_ingresso', 'forma_ingresso', 'faixa_membros_familia', 'ano_ingresso']

discente_limpo = discentes.copy()[colunas_discente].drop_duplicates()

In [199]:
discente_limpo.head()

,id_discente,status_discente,sexo,ano_nascimento,estado_civil,raca_declarada,id_curso,id_estrutura_curricular,media_geral,ch_integralizada,ch_pendente,faixa_renda_familiar,periodo_ingresso,forma_ingresso,faixa_membros_familia,ano_ingresso
0,1169287803480640,CANCELADO,M,1994.0,Solteiro(a),Nao_informado,2558681,25587020,0.93,68.0,1432.0,nulo,2,PROCESSO SELETIVO,2-3,2016
1,8959954670178323,CANCELADO,F,1997.0,Solteiro(a),Nao_informado,1958830,23647550,0.00,0.0,0.0,nulo,2,PROCESSO SELETIVO,2-3,2016
2,9718041069456731,CONCLUÍDO,F,1970.0,Solteiro(a),Negra,31496451,319171030,9.05,240.0,0.0,nulo,1,PROCESSO SELETIVO,2-3,2023
3,9657337360405103,CONCLUÍDO,F,1995.0,Outro,Negra,3926060,39260790,9.00,1200.0,0.0,ate_1k,2,PROCESSO SELETIVO,2-3,2018
4,8032369354468776,CONCLUÍDO,F,2000.0,Solteiro(a),Negra,2663867,29380940,7.27,1305.0,0.0,nulo,1,PROCESSO SELETIVO,2-3,2018


### 3.2. Tabela de docentes (agregação departamento × ano)

- **Tipos** — `ano_admissao` e `ano_desligamento` numéricos.
- **Proporções** — por sexo feminino e por tipo de vínculo (DE, T40, T20), calculadas por **professor** e depois **médias** por grupo.
- **Grade cartesiana** — pares (centro, departamento) × anos presentes em `matriculas`; filtra docentes **ativos** no ano (`admissão ≤ ano` e `desligamento` ausente ou posterior).
- **`docentes_limpo`** — contagem de docentes (`total_docentes`), proporções e mediana de ano de admissão por grupo.

> **Nota:** Neste notebook, o DataFrame principal após os joins **não** inclui `docentes_limpo` por linha de matrícula (evita duplicar lógica de teste vs. produção). A agregação fica pronta para inclusão futura com `merge(..., on=['sigla_centro','sigla_departamento','ano'], validate='m:1')`.

In [200]:
docentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pessoa           348 non-null    object 
 1   sexo                348 non-null    object 
 2   tipo_vinculo        348 non-null    object 
 3   sigla_departamento  348 non-null    object 
 4   nome_departamento   348 non-null    object 
 5   sigla_centro        348 non-null    object 
 6   nome_centro         348 non-null    object 
 7   situacao            348 non-null    object 
 8   cargo               348 non-null    object 
 9   ano_admissao        342 non-null    float64
 10  ano_desligamento    217 non-null    float64
dtypes: float64(2), object(9)
memory usage: 30.0+ KB


In [201]:
doc = docentes.copy()
doc['ano_admissao'] = pd.to_numeric(doc['ano_admissao'], errors='coerce')
doc['ano_desligamento'] = pd.to_numeric(doc['ano_desligamento'], errors='coerce')

In [202]:
agg_map = {'id_pessoa': 'count', 'ano_admissao': 'median'}
if 'sexo' in doc.columns:
  doc['_fem'] = (doc['sexo'] == 'F').astype(int)
  agg_map['_fem'] = 'mean'
if 'tipo_vinculo' in doc.columns:
  VINCULO_MAP = {
      'DE': 'DE (Dedicação Exclusiva)',
      'T40': 'T40 (40 horas semanais)',
      'T20': 'T20 (20 horas semanais)',
  }
  tipo_vinculo_normalizado = doc['tipo_vinculo'].astype(str).str.strip().str.upper()
  doc['tipo_vinculo_label'] = tipo_vinculo_normalizado.map(VINCULO_MAP).fillna(
      doc['tipo_vinculo'].astype(str)
  )
  for codigo_vinculo in VINCULO_MAP:
      doc[f'_vinc_{codigo_vinculo}'] = (tipo_vinculo_normalizado == codigo_vinculo).astype(int)
      agg_map[f'_vinc_{codigo_vinculo}'] = 'mean'

colunas_para_espelhar = ['_fem', '_vinc_DE', '_vinc_T40', '_vinc_T20', 'tipo_vinculo_label']
for coluna in colunas_para_espelhar:
  if coluna in doc.columns:
      docentes[coluna] = doc[coluna].values


In [203]:
ANOS_REF = list(matriculas['ano'].unique())
colunas_geograficas = [c for c in ('sigla_centro', 'sigla_departamento') if c in doc.columns]
if 'sigla_departamento' not in colunas_geograficas:
  raise ValueError('A base de docentes precisa da coluna sigla_departamento.')
pares_centro_departamento = doc[colunas_geograficas].drop_duplicates()
grade_ano_referencia = pares_centro_departamento.merge(
  pd.DataFrame({'ano': ANOS_REF}), how='cross'
)
grade_ano_referencia['ano'] = pd.to_numeric(grade_ano_referencia['ano'], errors='coerce')


chaves_merge_doc = [
  c for c in ('sigla_departamento', 'sigla_centro')
  if c in doc.columns and c in grade_ano_referencia.columns
]
grade_com_cadastro_docente = grade_ano_referencia.merge(doc, on=chaves_merge_doc, how='inner')

ano_admissao = grade_com_cadastro_docente['ano_admissao']
ano_desligamento = grade_com_cadastro_docente['ano_desligamento']
ano_referencia = grade_com_cadastro_docente['ano']
esta_ativo = (
  ano_admissao.notna()
  & (ano_admissao <= ano_referencia)
  & (ano_desligamento.isna() | (ano_desligamento > ano_referencia))
)
docentes_ativos = grade_com_cadastro_docente.loc[esta_ativo]

chaves_agregacao = [
  c for c in ('sigla_centro', 'sigla_departamento', 'ano')
  if c in docentes_ativos.columns
]
docentes_limpo = (
  docentes_ativos.groupby(chaves_agregacao, as_index=False, dropna=False)
  .agg(agg_map)
  .rename(columns={
      'id_pessoa': 'total_docentes',
      '_fem': 'prop_feminino',
      '_vinc_DE': 'prop_de',
      '_vinc_T40': 'prop_t40',
      '_vinc_T20': 'prop_t20',
      'ano_admissao': 'ano_admissao_medio',
  })
)
docentes_limpo

,sigla_centro,sigla_departamento,ano,total_docentes,ano_admissao_medio,prop_feminino,prop_de,prop_t40,prop_t20
0,CCA,CCA-DCFS,2004,1,1984.0,1.0,1.0,0.0,0.0
1,CCA,CCA-DCFS,2005,1,1984.0,1.0,1.0,0.0,0.0
2,CCA,CCA-DCFS,2006,1,1984.0,1.0,1.0,0.0,0.0
3,CCA,CCA-DCFS,2009,1,1984.0,1.0,1.0,0.0,0.0
4,CCA,CCA-DCFS,2011,1,1984.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
349,UFPB.,CS,2011,1,2007.0,1.0,1.0,0.0,0.0
350,UFPB.,CS,2012,1,2007.0,1.0,1.0,0.0,0.0
351,UFPB.,PROEX,2022,1,2022.0,0.0,1.0,0.0,0.0
352,UFPB.,PROEX,2023,1,2022.0,0.0,1.0,0.0,0.0


### 3.3. Componentes curriculares

- Inspeção de categorias (`info`, tipos de componente, nível, vagas).
- **`area_conhecimento`** — via `mapeamento_disciplinas.json` (`dicionario_areas`).
- **Percentuais de CH** — aula, EAD, laboratório sobre `ch_total` (evitando divisão por zero).
- **`tipo_disciplina_carga`** — classifica como predominante **Aula**, **EAD**, **Laboratório** ou **Mista** com limiares de domínio (`TH_DOM`, `TH_GAP`).
- **`componentes_limpa`** — colunas finais + deduplicação.

In [204]:
componentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 22 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   id_disciplina                         998 non-null    int64 
 1   id_detalhe                            998 non-null    int64 
 2   nome                                  998 non-null    object
 3   ch_aula                               998 non-null    int64 
 4   ch_laboratorio                        998 non-null    int64 
 5   ch_total                              998 non-null    int64 
 6   cr_aula                               998 non-null    int64 
 7   cr_laboratorio                        998 non-null    int64 
 8   cr_estagio                            998 non-null    int64 
 9   ch_ead                                998 non-null    int64 
 10  sigla_departamento                    998 non-null    object
 11  nivel_componente_curricular           998 n

In [205]:
componentes['descricao_tipo_componente_curricular'].unique()

array(['DISCIPLINA', 'ATIVIDADE', 'MÓDULO'], dtype=object)

In [206]:
componentes['qtd_max_matriculas'].unique()

array([1])

In [207]:
componentes['nivel_componente_curricular'].unique()

array(['T'], dtype=object)

In [208]:
with open("mapeamento_disciplinas.json", encoding="utf-8") as f:
    dados = json.load(f)

dicionario_areas = dados["dicionario_areas"]

componentes["area_conhecimento"] = componentes["nome_componete_curricular"].map(dicionario_areas)

componentes['pct_carga_horaria_lab'] = componentes['ch_laboratorio']/componentes['ch_total']
componentes['pct_carga_horaria_ead'] = componentes['ch_ead']/componentes['ch_total']
componentes['pct_carga_horaria_aula'] = componentes['ch_aula']/componentes['ch_total']


den = componentes["ch_total"].replace(0, np.nan)

p_aula = componentes["pct_carga_horaria_aula"].fillna(0.0)
p_ead = componentes["pct_carga_horaria_ead"].fillna(0.0)
p_lab = componentes["pct_carga_horaria_lab"].fillna(0.0)

vals = np.column_stack([p_aula.to_numpy(), p_ead.to_numpy(), p_lab.to_numpy()])
nomes = np.array(["Aula", "EAD", "Laboratório"])

s = np.sort(vals, axis=1)
topv = s[:, -1]
second = s[:, -2]
j = np.argmax(vals, axis=1)

TH_DOM = 0.50
TH_GAP = 0.10

puro = (topv >= TH_DOM) & ((topv - second) > TH_GAP)
componentes["tipo_disciplina_carga"] = np.where(puro, nomes[j], "Mista")

In [209]:
colunas_componentes = ['id_disciplina', 'codigo_componente_curricular', 'area_conhecimento', 'ch_aula', 'ch_ead',
                        'ch_laboratorio', 'ch_total','pct_carga_horaria_lab', 'pct_carga_horaria_ead', 'pct_carga_horaria_aula', 'sigla_departamento', 'sigla_centro',
                        'tipo_disciplina_carga', 'descricao_tipo_componente_curricular', 'nome_componete_curricular']

In [210]:
componentes_limpa = componentes.copy()[colunas_componentes].drop_duplicates()

In [211]:
componentes_limpa.head()

,id_disciplina,codigo_componente_curricular,area_conhecimento,ch_aula,ch_ead,ch_laboratorio,ch_total,pct_carga_horaria_lab,pct_carga_horaria_ead,pct_carga_horaria_aula,sigla_departamento,sigla_centro,tipo_disciplina_carga,descricao_tipo_componente_curricular,nome_componete_curricular
0,18066,TETS0001,Ciências da Saúde,30,0,0,30,0.00,0.0,1.00,CPT-ETS-COAGCEP,CPT-ETS,Aula,DISCIPLINA,NOÇÕES DE PRIMEIROS SOCORROS
1,18069,TETS0002,Ciências Biológicas,40,0,0,40,0.00,0.0,1.00,CPT-ETS-COAGCEP,CPT-ETS,Aula,DISCIPLINA,NOÇÕES DE ANATOMIA E FISIOLOGIA HUMANA
2,18070,TETS0003,Ciências Biológicas,10,0,30,40,0.75,0.0,0.25,CPT-ETS-COAGCEP,CPT-ETS,Laboratório,DISCIPLINA,"NOÇÕES DE MICROBIOLOGIA, PARASITOLOGIA E IMUNO..."
3,18071,TETS0004,Ciências da Saúde,65,0,0,65,0.00,0.0,1.00,CPT-ETS-COAGCEP,CPT-ETS,Aula,DISCIPLINA,EDUCAÇÃO EM SAÚDE
4,18072,TETS0005,Ciências Humanas e Sociais Aplicadas,50,0,0,50,0.00,0.0,1.00,CPT-ETS-COAGCEP,CPT-ETS,Aula,DISCIPLINA,PSICOLOGIA APLICADA À SAÚDE


### 3.4. Cursos

Seleção das colunas que ligam **curso × estrutura × disciplina** e expõem `carga_horaria` total do curso (usada nas features de proporção de CH integralizada).

In [212]:
cursos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1310 entries, 0 to 1309
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id_curso                        1310 non-null   int64  
 1   curso_nome                      1310 non-null   object 
 2   curso_unidade_nome              1310 non-null   object 
 3   campus                          1310 non-null   object 
 4   curso_ativo                     1310 non-null   bool   
 5   curso_grande_area_conhecimento  0 non-null      float64
 6   id_estrutura_curricular         1310 non-null   int64  
 7   codigo_emec                     0 non-null      float64
 8   turno_estrutura_curricular      1310 non-null   object 
 9   periodo                         1310 non-null   object 
 10  estrutura_curricular_ativo      1310 non-null   bool   
 11  carga_horaria                   1310 non-null   int64  
 12  id_modulo                       1310 non-null

In [213]:
cursos['carga_horaria'].unique()

array([2000, 1800, 4035, 2790, 1545, 4045, 1555, 1310, 1305, 1275,  800,
        945,  160, 1200, 1050, 3360, 3375, 1365, 1185, 1500, 1350, 1605,
       1110, 1410, 1245,  165,  810, 1345, 1495, 1320])

In [214]:
cursos_limpo = cursos.copy()[['id_curso', 'curso_nome', 'id_estrutura_curricular', 'id_disciplina', 'carga_horaria']]

### 3.5. Junção no nível da matrícula

Ordem dos merges (todos `left` a partir de `matriculas`):

1. `discente_limpo` em `id_discente` (`m:1`).
2. `componentes_limpa` em `id_disciplina` (`m:1`).
3. `cursos_limpo` em `id_curso`, `id_estrutura_curricular`, `id_disciplina` (`m:1`).

`ano` é forçado a numérico para consistência. Registros **sem sexo** do discente são removidos (`dropna(subset='sexo')`) por dependência de features demográficas.

Opcional: acrescentar docentes com `df.merge(docentes_limpo, on=['sigla_centro','sigla_departamento','ano'], how='left', validate='m:1')`.

In [215]:
matriculas['ano'] = pd.to_numeric(matriculas['ano'], errors="coerce")
docentes_limpo["ano"] = pd.to_numeric(docentes_limpo["ano"], errors="coerce")

df = (
  matriculas.merge(discente_limpo, on="id_discente", how="left", validate="m:1")
  .merge(componentes_limpa, on="id_disciplina", how="left", validate="m:1")
  .merge(cursos_limpo, on=["id_curso", "id_estrutura_curricular", "id_disciplina"], how="left", validate="m:1")
).dropna(subset='sexo')

df = df.dropna(subset='curso_nome')


### limitando a quantidade de anos de dados

In [216]:
df = df.copy().query("ano >= 2015")

## 4. Definição da variável-alvo

| Valor | Classe | Situações (exemplos) |
|-------|--------|----------------------|
| **1** | Sucesso | APROVADO, APROV. C/ DISTINÇÃO, DISPENSADO, APROVEITADO |
| **0** | Insucesso | REPROVADO, REP. FALTA, CANCELADO, TRANCADO, etc. |

Situações ainda em curso (ex.: **MATRICULADO**) são **excluídas** do conjunto de modelagem, pois não há desfecho fechado.


In [217]:
df['situacao'].value_counts()

situacao
APROVADO               93719
REPROVADO              13605
EXCLUIDA                9618
REP. MEDIA E FALTA      9002
MATRICULADO             6201
TRANCADO                4364
APROVEITADO             4060
CANCELADO               2527
DISPENSADO               469
DESISTENCIA              273
REP. FALTA               265
TRANSFERIDO              205
APROV. C/ DISTINÇÃO        7
INDEFERIDO                 2
Name: count, dtype: int64

In [218]:
df.query("situacao == 'EXCLUIDA'")['status_discente'].unique()

array(['CANCELADO', 'CONCLUÍDO', 'ATIVO', 'ATIVO - FORMANDO', 'TRANCADO'],
      dtype=object)

### da para saber mais ou menos em que periodo ele largou crruzando situacao x status_discente

In [219]:
df.query("id_discente == '0014367280169433'")

,id_discente,id_disciplina,ano,periodo,situacao,status_discente,sexo,ano_nascimento,estado_civil,raca_declarada,...,pct_carga_horaria_lab,pct_carga_horaria_ead,pct_carga_horaria_aula,sigla_departamento,sigla_centro,tipo_disciplina_carga,descricao_tipo_componente_curricular,nome_componete_curricular,curso_nome,carga_horaria
241745,0014367280169433,18678,2017,1,REP. MEDIA E FALTA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,APICULTURA,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241746,0014367280169433,18689,2017,2,REPROVADO,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,GESTÃO E EXTENSÃO RURAL,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241747,0014367280169433,18682,2017,2,EXCLUIDA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,CONSTRUÇÕES RURAIS,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241748,0014367280169433,18672,2017,1,REP. MEDIA E FALTA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,SOLOS,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241749,0014367280169433,18679,2017,1,REPROVADO,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,CUNICULTURA,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241750,0014367280169433,18668,2017,1,REP. MEDIA E FALTA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,FITOSSANIDADE,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241751,0014367280169433,18673,2017,1,REP. MEDIA E FALTA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,IRRIGAÇÃO E DRENAGEM,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241752,0014367280169433,18680,2017,1,REP. MEDIA E FALTA,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,RANICULTURA,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241753,0014367280169433,18669,2017,1,REPROVADO,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,SILVICULTURA,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0
241754,0014367280169433,18675,2017,1,REPROVADO,CANCELADO,M,1987.0,Casado(a),Negra,...,0.333333,0.0,0.666667,CCHSA - CAVN,CCHSA,Aula,DISCIPLINA,MECANIZAÇÃO AGRÍCOLA,TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...,1545.0


In [220]:
SITUACOES_SUCESSO = [
    'APROVADO',
    'APROV. C/ DISTINÇÃO',
    'DISPENSADO',
    'APROVEITADO'
]
SITUACOES_INSUCESSO = [
    'REPROVADO',
    'REP. MEDIA E FALTA',
    'REP. FALTA',
    'TRANCADO',
    'DESISTENCIA',
]

SITUACOES_EXCLUIR_DO_MODELO = [
    'MATRICULADO',
    'INDEFERIDO',
    'EXCLUIDA',
    'TRANSFERIDO'
]

situacoes_finais = SITUACOES_SUCESSO + SITUACOES_INSUCESSO
df_modelo = df[df['situacao'].isin(situacoes_finais)].copy()
df_modelo['target'] = df_modelo['situacao'].isin(SITUACOES_SUCESSO).astype(int)

vc = df_modelo['target'].value_counts()
print(f'Total (situacoes finais): {len(df_modelo):,}')
print(f'Sucesso   (1): {vc.get(1,0):,} ({vc.get(1,0)/len(df_modelo)*100:.1f}%)')
print(f'Insucesso (0): {vc.get(0,0):,} ({vc.get(0,0)/len(df_modelo)*100:.1f}%)')
ratio = vc.get(1,0) / max(vc.get(0,0), 1)
print(f'Razão S/I: {ratio:.2f}' + (' ⚠️ desbalanceado' if ratio > 3 else ''))

Total (situacoes finais): 125,764
Sucesso   (1): 98,255 (78.1%)
Insucesso (0): 27,509 (21.9%)
Razão S/I: 3.57 ⚠️ desbalanceado


## 5. Engenharia de features (sem one-hot neste notebook)

### 5.1. Histórico por área e CH acumulada (sem vazamento)

Linhas são ordenadas por discente, **ano**, período extraído de `periodo` e `id_disciplina`. Para cada linha, acumula-se **somente o histórico anterior** na mesma **área de conhecimento**:

- contagens de aprovações/reprovações e CH aprovada/reprovada;
- **taxas** de sucesso por contagem e por carga horária (só entre aprov vs reprov, como antes);
- **trancamento/desistência** na mesma área: `n_tranc_desist_antes_mesma_area` e **taxa** `(tranc+desist) / (aprov+reprov+tranc+desist)` por contagem e por CH (`taxa_tranc_desist_*`), usando `TRANCADO` e `DESISTENCIA`;
- **CH integralizada** (somatório de CH em situações de sucesso antes da linha) e **proporção** em relação à `carga_horaria` do curso.

Assim, cada matrícula “enxerga” apenas o passado compatível com inferência preditiva.

In [221]:
df_modelo.info()

<class 'pandas.DataFrame'>
Index: 125764 entries, 11 to 426752
Data columns (total 37 columns):
 #   Column                                Non-Null Count   Dtype   
---  ------                                --------------   -----   
 0   id_discente                           125764 non-null  object  
 1   id_disciplina                         125764 non-null  int64   
 2   ano                                   125764 non-null  int64   
 3   periodo                               125764 non-null  object  
 4   situacao                              125764 non-null  object  
 5   status_discente                       125764 non-null  object  
 6   sexo                                  125764 non-null  object  
 7   ano_nascimento                        125731 non-null  float64 
 8   estado_civil                          125764 non-null  object  
 9   raca_declarada                        125764 non-null  object  
 10  id_curso                              125764 non-null  float64 
 11  id

In [222]:
SUCESSO = {
    "APROVADO",
    "APROVEITADO",
    "DISPENSADO",
    "APROV. C/ DISTINÇÃO",
}
REPROVA = {
    'REPROVADO',
    'REP. MEDIA E FALTA',
    'REP. FALTA'
}
TRANCADO_DESI = {
    'TRANCADO',
    'DESISTENCIA',
}


df = df_modelo.copy()
df["_po"] = pd.to_numeric(
    df["periodo"].astype(str).str.extract(r"(\d+)")[0], errors="coerce"
)
df = df.sort_values(
    ["id_discente", "ano", "_po", "id_disciplina"],
    kind="mergesort",
)

def _blank():
    return {
        "n_ok": 0, "n_rep": 0, "n_td": 0,
        "ch_ok": 0.0, "ch_rep": 0.0, "ch_td": 0.0,
    }

n_ok_l, n_rep_l, n_td_l = [], [], []
ch_ok_l, ch_rep_l, ch_td_l = [], [], []
taxa_n_l, taxa_ch_l = [], []
taxa_tranc_desist_n_l, taxa_tranc_desist_ch_l = [], []
ch_integralizado_l = []
ch_integralizado_prop_l = []

for _, g in df.groupby("id_discente", sort=False):
    por_area = {}
    ch_integralizado = 0.0
    ch_integralizado_prop = 0.0
    ch_total_curso = g["carga_horaria"].iloc[0] if "carga_horaria" in g.columns else np.nan
    for _, row in g.iterrows():
        area = row["area_conhecimento"]
        if pd.isna(area) or str(area).strip() == "":
            area = "__sem_area__"
        st = por_area.get(area, _blank())
        n_ok, n_rep, n_td = st["n_ok"], st["n_rep"], st["n_td"]
        ch_ok, ch_rep, ch_td = st["ch_ok"], st["ch_rep"], st["ch_td"]
        n_ok_l.append(n_ok)
        n_rep_l.append(n_rep)
        n_td_l.append(n_td)
        ch_ok_l.append(ch_ok)
        ch_rep_l.append(ch_rep)
        ch_td_l.append(ch_td)
        den_n = n_ok + n_rep
        taxa_n_l.append(n_ok / den_n if den_n else np.nan)
        den_ch = ch_ok + ch_rep
        taxa_ch_l.append(ch_ok / den_ch if den_ch > 0 else np.nan)
        den_tudo_n = n_ok + n_rep + n_td
        den_tudo_ch = ch_ok + ch_rep + ch_td
        taxa_tranc_desist_n_l.append(n_td / den_tudo_n if den_tudo_n > 0 else np.nan)
        taxa_tranc_desist_ch_l.append(ch_td / den_tudo_ch if den_tudo_ch > 0 else np.nan)

        ch_integralizado_l.append(ch_integralizado)
        if ch_total_curso and ch_total_curso > 0:
            ch_integralizado_prop_l.append(ch_integralizado / ch_total_curso)
        else:
            ch_integralizado_prop_l.append(np.nan)
        sit = row["situacao"]
        ch = float(row["ch_total"]) if pd.notna(row["ch_total"]) else 0.0
        if sit in SUCESSO:
            st["n_ok"] += 1
            st["ch_ok"] += ch
            ch_integralizado += ch
        elif sit in REPROVA:
            st["n_rep"] += 1
            st["ch_rep"] += ch
        elif sit in TRANCADO_DESI:
            st["n_td"] += 1
            st["ch_td"] += ch
        por_area[area] = st

df["n_aprov_antes_mesma_area"] = n_ok_l
df["n_reprov_antes_mesma_area"] = n_rep_l
df["n_tranc_desist_antes_mesma_area"] = n_td_l
df["taxa_sucesso_count_antes_mesma_area"] = taxa_n_l
df["taxa_sucesso_ch_antes_mesma_area"] = taxa_ch_l
df["taxa_tranc_desist_count_antes_mesma_area"] = taxa_tranc_desist_n_l
df["taxa_tranc_desist_ch_antes_mesma_area"] = taxa_tranc_desist_ch_l

df["ch_integralizada_atual"] = ch_integralizado_l
df["progresso_atual_curso"] = ch_integralizado_prop_l

_cols_hist_area = [
    "n_aprov_antes_mesma_area",
    "n_reprov_antes_mesma_area",
    "n_tranc_desist_antes_mesma_area",
    "taxa_sucesso_count_antes_mesma_area",
    "taxa_sucesso_ch_antes_mesma_area",
    "taxa_tranc_desist_count_antes_mesma_area",
    "taxa_tranc_desist_ch_antes_mesma_area",
    "ch_integralizada_atual",
    "progresso_atual_curso",
]
df[_cols_hist_area] = df[_cols_hist_area].fillna(0)

df = df.drop(columns=["_po"], errors="ignore")

df_modelo_tratado = df


### 5.2. Demografia e indicadores auxiliares

- **`primeiro_periodo`**: linhas cujo `ano` coincide com `ano_ingresso` (marco de entrada).
- **`idade`**: diferença `ano - ano_nascimento`; outliers IQR são tratados como na tabela de discentes, com imputação pela **mediana**.

In [223]:
df_modelo_tratado["primeiro_periodo"] = (df_modelo_tratado["ano"].astype(float) == df_modelo_tratado["ano_ingresso"].astype(float)).astype(np.int8)

In [224]:
df_modelo_tratado['idade'] = df_modelo_tratado['ano'] - df_modelo_tratado['ano_nascimento']

k = 1.5
colunas = ["idade"]

for col in colunas:
    if col not in df_modelo_tratado.columns:
        continue
    mask = mascara_outliers_iqr(df_modelo_tratado[col], k=k, apenas_superior=False)
    mask = mask.reindex(df_modelo_tratado.index).fillna(False)
    df_modelo_tratado.loc[mask, col] = np.nan

for col in colunas:
    if col in df_modelo_tratado.columns:
        df_modelo_tratado[col] = df_modelo_tratado[col].fillna(df_modelo_tratado[col].median())

Conferência final do esquema antes de exportar (tipos, contagem de colunas e nulos).

In [225]:
df_modelo_tratado.info()

<class 'pandas.DataFrame'>
Index: 125764 entries, 254385 to 122187
Data columns (total 48 columns):
 #   Column                                    Non-Null Count   Dtype   
---  ------                                    --------------   -----   
 0   id_discente                               125764 non-null  object  
 1   id_disciplina                             125764 non-null  int64   
 2   ano                                       125764 non-null  int64   
 3   periodo                                   125764 non-null  object  
 4   situacao                                  125764 non-null  object  
 5   status_discente                           125764 non-null  object  
 6   sexo                                      125764 non-null  object  
 7   ano_nascimento                            125731 non-null  float64 
 8   estado_civil                              125764 non-null  object  
 9   raca_declarada                            125764 non-null  object  
 10  id_curso           

In [226]:
df['curso_nome'].unique()

array(['TÉCNICO DE NÍVEL MÉDIO EM AGROINDÚSTRIA NA FORMA INTEGRADA',
       'TÉCNICO EM PRÓTESE DENTÁRIA',
       'TÉCNICO EM REGISTROS E INFORMAÇÕES EM SAÚDE - EaD',
       'TÉCNICO DE NÍVEL MÉDIO EM NUTRIÇÃO E DIETÉTICA NA FORMA SUBSEQUENTE',
       'TÉCNICO EM ENFERMAGEM',
       'TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORMA SUBSEQUENTE',
       'TÉCNICO DE NÍVEL MÉDIO EM AGROINDÚSTRIA NA FORMA SUBSEQUENTE',
       'TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORMA INTEGRADA',
       'TÉCNICO EM CUIDADOS DE IDOSOS', 'TÉCNICO EM ANÁLISES CLÍNICAS',
       'TÉCNICO DE NÍVEL MÉDIO EM AQUICULTURA NA FORMA SUBSEQUENTE',
       'TÉCNICO DE NÍVEL MÉDIO EM VETERINÁRIA NA FORMA SUBSEQUENTE',
       'TÉCNICO EM CUIDADOS DE IDOSOS - PROEJA',
       'Formação Inicial e Continuada:Língua Portuguesa e Cultura Brasileira para Estrangeiros - Básico',
       'TÉCNICO EM SAÚDE BUCAL', 'TÉCNICO EM CITOPATOLOGIA',
       'TÉCNICO DE NÍVEL MÉDIO EM PAISAGISMO NA FORMA SUBSEQUENTE',
       'TÉCNICO EM A

## 6. Exportação dos dados tratados

Arquivo: **`dados/df_modelo_tratado.csv`**. Em seguida, execute `02_analise_exploratoria.ipynb` e `03_modelagem_previsao.ipynb`.

In [227]:
OUTPUT_CSV = DATA_PATH + 'df_modelo_tratado.csv'
df_modelo_tratado.to_csv(OUTPUT_CSV, index=False)
print(f'Salvo: {OUTPUT_CSV} | linhas={len(df_modelo_tratado):,} | colunas={df_modelo_tratado.shape[1]}')


Salvo: dados/df_modelo_tratado.csv | linhas=125,764 | colunas=48
